# Spatial analysis of green cover and the built environment

This notebook quantifies the areas removed from the combined urban-green-area and conservation-land baseline by land-cover class 0 and building footprints. Area calculations use WGS 84 / UTM zone 14N (EPSG:32614).


In [ ]:
from pathlib import Path

import geopandas as gpd
import pandas as pd

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GREEN_AREAS_PATH = DATA_DIR / "areas_verdes.gpkg"
CONSERVATION_LAND_PATH = DATA_DIR / "suelo_de_conservacion.gpkg"
CLASSIFICATION_PATH = OUTPUT_DIR / "land_cover_classification.gpkg"
BUILDINGS_PATH = DATA_DIR / "edificios_cdmx.gpkg"

AREA_CRS = "EPSG:32614"

required_files = [
    GREEN_AREAS_PATH,
    CONSERVATION_LAND_PATH,
    CLASSIFICATION_PATH,
    BUILDINGS_PATH,
]
missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError(
        "The following required files were not found:
- " + "
- ".join(missing)
    )


In [ ]:
def load_and_prepare(path: Path, target_crs: str) -> gpd.GeoDataFrame:
    """Load a vector layer, validate its CRS, reproject, and repair geometries."""
    layer = gpd.read_file(path)
    if layer.crs is None:
        raise ValueError(
            f"{path.name} has no defined CRS. Define its original CRS before running the analysis."
        )
    layer = layer.to_crs(target_crs)
    layer = layer[~layer.geometry.is_empty & layer.geometry.notna()].copy()
    layer["geometry"] = layer.geometry.make_valid()
    return layer

areas_verdes = load_and_prepare(GREEN_AREAS_PATH, AREA_CRS)
suelo_conservacion = load_and_prepare(CONSERVATION_LAND_PATH, AREA_CRS)
clasificacion = load_and_prepare(CLASSIFICATION_PATH, AREA_CRS)
edificios = load_and_prepare(BUILDINGS_PATH, AREA_CRS)

print("Loaded layers:")
print(f"- Urban green areas: {len(areas_verdes):,}")
print(f"- Conservation land: {len(suelo_conservacion):,}")
print(f"- Land-cover polygons: {len(clasificacion):,}")
print(f"- Building footprints: {len(edificios):,}")


In [ ]:
# Combine urban green areas and conservation land into a single baseline geometry.
base = gpd.GeoDataFrame(
    geometry=pd.concat(
        [areas_verdes.geometry, suelo_conservacion.geometry],
        ignore_index=True,
    ),
    crs=AREA_CRS,
)
base_union = base.dissolve().reset_index(drop=True)
area_initial_ha = base_union.geometry.area.sum() / 10_000

# Select and dissolve class 0.
class_field = "class_id" if "class_id" in clasificacion.columns else "clase"
clasificacion[class_field] = clasificacion[class_field].astype(int)
class_0 = clasificacion.loc[clasificacion[class_field] == 0].copy()
if class_0.empty:
    raise ValueError("No polygons with land-cover class 0 were found.")
class_0_union = class_0.dissolve().reset_index(drop=True)

# Remove class 0 from the baseline.
remaining_after_class_0 = gpd.overlay(
    base_union,
    class_0_union,
    how="difference",
    keep_geom_type=True,
)
area_after_class_0_ha = remaining_after_class_0.geometry.area.sum() / 10_000

# Remove building footprints from the remaining geometry.
buildings_union = edificios.dissolve().reset_index(drop=True)
remaining_final = gpd.overlay(
    remaining_after_class_0,
    buildings_union,
    how="difference",
    keep_geom_type=True,
)
area_final_ha = remaining_final.geometry.area.sum() / 10_000


In [ ]:
area_removed_class_0_ha = area_initial_ha - area_after_class_0_ha
area_removed_buildings_ha = area_after_class_0_ha - area_final_ha
area_removed_total_ha = area_initial_ha - area_final_ha

results = pd.DataFrame(
    {
        "metric": [
            "Initial baseline area",
            "Area removed by class 0",
            "Area remaining after class 0",
            "Area removed by buildings",
            "Total area removed",
            "Final remaining area",
        ],
        "area_ha": [
            area_initial_ha,
            area_removed_class_0_ha,
            area_after_class_0_ha,
            area_removed_buildings_ha,
            area_removed_total_ha,
            area_final_ha,
        ],
    }
)
results["percentage_of_initial"] = results["area_ha"] / area_initial_ha * 100
results.round({"area_ha": 2, "percentage_of_initial": 2})


In [ ]:
# Store all vector outputs in one GeoPackage and the summary as CSV.
OUTPUT_GPKG = OUTPUT_DIR / "green_cover_spatial_analysis.gpkg"
OUTPUT_CSV = OUTPUT_DIR / "green_cover_area_summary.csv"

base_union.to_file(OUTPUT_GPKG, layer="baseline", driver="GPKG")
remaining_after_class_0.to_file(
    OUTPUT_GPKG,
    layer="remaining_after_class_0",
    driver="GPKG",
)
remaining_final.to_file(
    OUTPUT_GPKG,
    layer="final_remaining_area",
    driver="GPKG",
)
results.to_csv(OUTPUT_CSV, index=False)

print(f"Vector results: {OUTPUT_GPKG}")
print(f"Statistical summary: {OUTPUT_CSV}")
